In [ ]:
!nvidia-smi
import os
os.getcwd()

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# imports
# ──────────────────────────────────────────────────────────────────────────────
import glob
import os
import random
import re
import shutil
import time
import warnings
from collections import defaultdict
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchmetrics import Accuracy, F1Score
from torchvision import models
from torchsummary import summary
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    cohen_kappa_score
)
os.getcwd()
print(torch.__version__)

In [ ]:
# Set Seeds for Full Reproducibility and remove warnings

def set_seed(seed):
    """
    Set the random seed and enable A100‐optimized performance settings.
    """
    # 1. Python random
    random.seed(seed)
    # 2. NumPy
    np.random.seed(seed)
    # 3. PyTorch (CPU and GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 4. Enable TF32 for float32 matmuls on A100 (max throughput, minimal precision loss)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('medium')

    # 5. Enable cuDNN autotuner for best convolution kernels on A100
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

# SLURM warning from lightning_fabric
warnings.filterwarnings(
    "ignore",
    message="The `srun` command is available on your system but is not used"
)
# ModelCheckpoint registry warning
warnings.filterwarnings(
    "ignore",
    message="Using default `ModelCheckpoint`"
)
# log_every_n_steps warning
warnings.filterwarnings(
    "ignore",
    message="The number of training batches.*log_every_n_steps"
)

In [ ]:
# =============================================================================
# 2. Dataset + DataModule
# =============================================================================
import cv2
import numpy as np
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Union

import albumentations as A
from albumentations.pytorch import ToTensorV2
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch


class CXRDataset(Dataset):
    """
    - Train/Val: returns (image, label)
    - Test:      returns (image, label, filename) when return_paths=True
    """
    def __init__(
        self,
        file_paths: List[Path],
        labels: List[int],
        transform: Optional[A.Compose] = None,
        return_paths: bool = False
    ) -> None:
        assert len(file_paths) == len(labels), "file_paths and labels must align"
        self.file_paths  = file_paths
        self.labels      = labels
        self.transform   = transform
        self.return_paths = return_paths

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> Union[Tuple[torch.Tensor, int],
                                             Tuple[torch.Tensor, int, str]]:
        # always read as grayscale and expand to 3-channel RGB for pretrained backbones
        img = cv2.imread(str(self.file_paths[idx]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {self.file_paths[idx]}")
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        y = int(self.labels[idx])
        if self.return_paths:
            return img, y, self.file_paths[idx].name
        return img, y


class DataModule(pl.LightningDataModule):
    def __init__(self, h: Dict, fold_idx: Optional[int] = None) -> None:
        super().__init__()
        self.h = h
        self.fold_idx = fold_idx

        # Class definitions (index 1 == 'normal_not' == positive)
        self.class_names  = ["normal", "normal_not"]
        self.class_to_idx = {n: i for i, n in enumerate(self.class_names)}
        print(f"[DataModule] Class to index mapping: {self.class_to_idx}")

        # ---- Transforms ----
        size = int(h["img_size"])
        # Train (with CLAHE)
        self.train_transform = A.Compose([
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
            A.Resize(size, size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        # Val/Test (no augmentation)
        self.val_transform = A.Compose([
            A.Resize(size, size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.test_transform = self.val_transform

        # DataLoader args (avoid persistent_workers when num_workers == 0)
        num_workers = int(h["num_workers"])
        self.loader_args = dict(
            batch_size=int(h["batch_size"]),
            num_workers=num_workers,
            pin_memory=bool(h["pin_memory"]),
            persistent_workers=bool(num_workers > 0),
            prefetch_factor=2 if num_workers > 0 else None,
            drop_last=False
        )
        # torch DataLoader disallows None prefetch_factor; remove key if None
        if self.loader_args["prefetch_factor"] is None:
            self.loader_args.pop("prefetch_factor")

        # Expose test filenames for downstream alignment (filled in setup)
        self.test_filenames: List[str] = []

    def _make_split(self, base: Path, split: str) -> Tuple[List[Path], List[int]]:
        """
        Reads images from either:
          - base/<split>/<class>/   when split is non-empty, or
          - base/<class>/           when split == "".
        Ensures a deterministic, filename-sorted order across classes.
        """
        fps, labs = [], []
        for cls in self.class_names:
            folder = (base / split / cls) if split else (base / cls)
            if not folder.exists():
                continue
            imgs: List[Path] = []
            for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"):
                imgs.extend(folder.glob(ext))
            # sort deterministically by filename
            imgs = sorted(imgs, key=lambda p: p.name.lower())
            fps.extend(imgs)
            labs.extend([self.class_to_idx[cls]] * len(imgs))
        # final global sort by filename to make cross-class order stable as well
        if fps:
            order = np.argsort([p.name.lower() for p in fps])
            fps = [fps[i] for i in order]
            labs = [labs[i] for i in order]
        return fps, labs

    def setup(self, stage: Optional[str] = None) -> None:
        # ---- Train/Val for cross-validation ----
        if self.fold_idx is not None:
            base = Path(self.h["output_dir"]) / f"fold_{self.fold_idx}"
            tr_fps, tr_labs = self._make_split(base, "train")
            vl_fps, vl_labs = self._make_split(base, "val")

            if self.h.get("balance", False) and len(tr_labs) > 0:
                counts = np.bincount(tr_labs, minlength=len(self.class_names))
                weights = (1.0 / np.maximum(counts, 1))[np.array(tr_labs)]
                weights = torch.as_tensor(weights, dtype=torch.double)
                self.train_sampler = WeightedRandomSampler(
                    weights, num_samples=len(weights), replacement=True
                )
                print("[DataModule] Using WeightedRandomSampler to correct class-imbalance")
            else:
                self.train_sampler = None
                print("[DataModule] Using standard shuffling (no imbalance correction)")

            self.train_dataset = CXRDataset(tr_fps, tr_labs, transform=self.train_transform, return_paths=False)
            self.val_dataset   = CXRDataset(vl_fps, vl_labs, transform=self.val_transform,   return_paths=False)
        else:
            self.train_dataset = None
            self.val_dataset   = None
            self.train_sampler = None
            print("[DataModule] Skipping train/val setup (fold_idx is None)")

        # ---- Always build TEST (shared across all ensemble members) ----
        td = Path(self.h["test_dir"])
        ts_fps, ts_labs = self._make_split(td, "")
        if not ts_fps:
            ts_fps, ts_labs = self._make_split(td, "test")
        self.test_dataset = CXRDataset(ts_fps, ts_labs, transform=self.test_transform, return_paths=True)
        self.test_filenames = [p.name for p in ts_fps]
        print(f"[DataModule] Test set: {len(ts_fps)} samples")

    def train_dataloader(self):
        if self.train_dataset is None:
            return None
        if self.train_sampler is not None:
            return DataLoader(
                self.train_dataset,
                sampler=self.train_sampler,
                shuffle=False,
                **self.loader_args
            )
        return DataLoader(
            self.train_dataset,
            shuffle=True,
            **self.loader_args
        )

    def val_dataloader(self):
        if self.val_dataset is None:
            return None
        return DataLoader(
            self.val_dataset,
            shuffle=False,
            **self.loader_args
        )

    def test_dataloader(self):
        # returns (image, label, filename) batches
        return DataLoader(
            self.test_dataset,
            shuffle=False,
            **self.loader_args
        )

In [ ]:
# ===========================
# TB_model_dpn68
# ===========================
from __future__ import annotations

from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import timm
from torchmetrics.classification import Accuracy, F1Score


# ---- helpers to find the last Conv2d that writes the final stage feature map ----
class _ConvSpy:
    def __init__(self) -> None:
        self.records: List[Tuple[str, nn.Module, Tuple[int, int, int, int]]] = []  # (name, mod, out.shape)
        self.hooks: List[torch.utils.hooks.RemovableHandle] = []

    def hook(self, name: str, mod: nn.Conv2d):
        def _h(m, x, y):
            if isinstance(y, torch.Tensor) and y.ndim == 4:
                self.records.append((name, mod, tuple(y.shape)))
        return _h

    def attach(self, module: nn.Module) -> None:
        for n, m in module.named_modules():
            if isinstance(m, nn.Conv2d):
                self.hooks.append(m.register_forward_hook(self.hook(n, m)))

    def detach(self) -> None:
        for h in self.hooks:
            try:
                h.remove()
            except Exception:
                pass
        self.hooks.clear()


def _discover_laststage_conv_k(backbone: nn.Module, img_size: int) -> Tuple[Optional[nn.Conv2d], Optional[Tuple[int, int]]]:
    """
    Returns (last_conv_in_final_stage, kernel_size) if any Conv2d writes the final stage map; else (None, None).
    We run on CPU with a 1x3xH×W dummy input to avoid GPU binding during Lightning __init__.
    """
    spy = _ConvSpy()
    spy.attach(backbone)

    with torch.no_grad():
        x = torch.zeros(1, 3, img_size, img_size, dtype=torch.float32)
        feats = backbone(x)  # timm features_only → list[tensor]
        if isinstance(feats, (list, tuple)):
            last = feats[-1]
        else:
            last = feats
        last_hw = tuple(last.shape[-2:])

    spy.detach()

    # keep convs that produced tensors with same H,W as final map
    same_hw: List[Tuple[str, nn.Module, Tuple[int, int, int, int]]] = []
    for (n, m, shp) in spy.records:
        if tuple(shp[-2:]) == last_hw:
            same_hw.append((n, m, shp))

    if not same_hw:
        return None, None

    # execution order ~ records order → take the last one
    _, last_conv, _ = same_hw[-1]
    k = last_conv.kernel_size if isinstance(last_conv.kernel_size, tuple) else (last_conv.kernel_size, last_conv.kernel_size)
    return last_conv, (int(k[0]), int(k[1]))


# ======================
# TBModel (dpn68)
# ======================
class TBModel(pl.LightningModule):
    """
    Cleaned version of your TBModel that supports ONLY:
      - h['model'] == 'dpn68_new'  (mapped to timm id 'dpn68.mx_in1k', features_only=True)

    The backbone path, optional post3x3, and head are identical to the original dpn68_new branch.
    """
    def __init__(self, h: Dict):
        super().__init__()
        self.h: Dict = h
        self.save_hyperparameters('h')

        name: str = h["model"]
        img_size: int = int(h["img_size"])
        drop_rate: float = float(h["dropout"])
        num_classes: int = int(h["num_classes"])
        self.num_classes = num_classes

        # ------------------ 1) Backbone (DPN-68 only) ------------------
        self.is_transformer: bool = False

        if name != "dpn68_new":
            raise ValueError(f"Unsupported model '{name}'. This cleaned module supports only 'dpn68_new'.")

        timm_id = "dpn68.mx_in1k"
        base = timm.create_model(timm_id, pretrained=True, features_only=True)
        self.backbone = base
        feat_dim = base.feature_info[-1]["num_chs"]

        # Discover the last conv that writes the final stage map; if not 3×3 → add one
        last_conv, k = _discover_laststage_conv_k(self.backbone, img_size)
        need_3x3 = (k is None) or (k != (3, 3))
        if need_3x3:
            self.post3x3 = nn.Sequential(
                nn.Conv2d(feat_dim, feat_dim, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(feat_dim),
                nn.SiLU(inplace=True),
            )
        else:
            self.post3x3 = nn.Identity()

        # ------------------ 2) Head (unchanged) ------------------
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # (B, C, H, W) → (B, C, 1, 1)
            nn.Flatten(),             # → (B, C)
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, num_classes),
        )

        # ------------------ 3) Metrics (unchanged) ------------------
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes, average="macro")
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes, average="macro")
        self.train_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.val_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        if isinstance(feats, (list, tuple)):
            feats = feats[-1]
        feats = self.post3x3(feats)
        logits = self.head(feats)
        return logits

    @staticmethod
    def _nll_from_onehot_logits(logits: torch.Tensor, labels_oh: torch.Tensor) -> torch.Tensor:
        logp = F.log_softmax(logits, dim=1)
        return -(labels_oh * logp).sum(dim=1).mean()

    def training_step(self, batch, batch_idx):
        imgs, labels = batch
        if labels.ndim == 1:
            labels_oh = F.one_hot(labels, num_classes=self.num_classes).float()
        else:
            labels_oh = labels.float()
        logits = self(imgs)
        loss = self._nll_from_onehot_logits(logits, labels_oh)
        preds = logits.argmax(dim=1)
        tgt = labels_oh.argmax(dim=1)
        self.log("train_loss", loss, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc(preds, tgt), on_epoch=True, prog_bar=True)
        self.log("train_f1", self.train_f1(preds, tgt), on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        imgs, labels = batch
        if labels.ndim == 1:
            labels_oh = F.one_hot(labels, num_classes=self.num_classes).float()
        else:
            labels_oh = labels.float()
        logits = self(imgs)
        loss = self._nll_from_onehot_logits(logits, labels_oh)
        preds = logits.argmax(dim=1)
        tgt = labels_oh.argmax(dim=1)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc(preds, tgt), on_epoch=True, prog_bar=True)
        self.log("val_f1", self.val_f1(preds, tgt), on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.h.get("lr", 5e-5),
            weight_decay=1e-4,
        )
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.h.get("max_epochs", 64)
        )
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "interval": "epoch"}}


# ------------------------------------------------------------------
# History callback (epoch-level) (unchanged)
# ------------------------------------------------------------------
class History(pl.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.train_losses, self.val_losses = [], []
        self.train_accs, self.val_accs = [], []

    def on_train_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        if "train_loss" in m:
            self.train_losses.append(m["train_loss"].item())
        if "train_acc" in m:
            self.train_accs.append(m["train_acc"].item())

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        if "val_loss" in m:
            self.val_losses.append(m["val_loss"].item())
        if "val_acc" in m:
            self.val_accs.append(m["val_acc"].item())

In [ ]:
# =============================================================================
# Config
# =============================================================================

h = {
    "model": "dpn68_new",
    "img_size": 224,
    "batch_size": 64,
    "num_workers": 2,
    "dropout": 0.3,
    "num_classes": 2,
    "pin_memory": True,
    "lr": 5e-5,
    "max_epochs": 64,
    "patience": 10,
    "balance": True,
    "n_splits": 5,
    "train_dir": "/path/to/fold_specific/train/directory",
    "val_dir":   "/path/to/fold_specific/val/directory",
    "test_dir": "/path/to/test/directory",
    "output_dir":  "/path/to/output/directory",
    
    # plotting options
    "cmap": "Greens",
    "alpha": 0.85,
    "text_color": "black",
    "text_font_size": 38,
    "xt_label_rotation": 0,
    "xt_label_size": 22,
    "yt_label_size": 22,
    "xlabel_font_size": 26,
    "ylabel_font_size": 26,
}
# ----------------------------
# User paths
# ----------------------------
CKPT_PATH = Path(
    "/path/to/model/weights.ckpt"
)

RAD_CSV = Path(
    "/path/to/model/rad_annotations.csv"
)

OUT_ROOT = Path(
    "/path/to/save/results"
)

In [ ]:
# Seamless CAM overlays for pediatric lateral chest radiographs.

from __future__ import annotations

import os
import sys
from pathlib import Path
import logging
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms

from pytorch_grad_cam import (
    GradCAM,
    ScoreCAM,
    GradCAMPlusPlus,
    AblationCAM,
    XGradCAM,
    LayerCAM,
    FullGrad,
    HiResCAM,
    EigenCAM,
    EigenGradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# --------------------------------------------------------------------------------------
# HPC cache locations
# --------------------------------------------------------------------------------------
# Configure these paths before importing Torch, torchvision, timm, transformers, or
# huggingface_hub. Hugging Face otherwise derives its token/cache locations from HOME,
# which may still point to the retired /gpfs/gsfs12 filesystem in a long-lived notebook.

HPC_USER_HOME = Path("/vf/users/username")
HPC_CACHE_ROOT = HPC_USER_HOME / ".cache"
HF_HOME_DIR = HPC_CACHE_ROOT / "huggingface"

def configure_hpc_caches() -> None:
    """Route model caches and the Hugging Face token to the current /vf filesystem."""
    cache_paths = {
        "XDG_CACHE_HOME": HPC_CACHE_ROOT,
        "HF_HOME": HF_HOME_DIR,
        "HF_HUB_CACHE": HF_HOME_DIR / "hub",
        # Compatibility alias used by older huggingface_hub releases.
        "HUGGINGFACE_HUB_CACHE": HF_HOME_DIR / "hub",
        "HF_TOKEN_PATH": HF_HOME_DIR / "token",
        "TRANSFORMERS_CACHE": HF_HOME_DIR / "transformers",
        "HF_DATASETS_CACHE": HF_HOME_DIR / "datasets",
        "TORCH_HOME": HPC_CACHE_ROOT / "torch",
    }

    # Assignment is intentional: setdefault() would retain stale /gpfs paths.
    for variable, path in cache_paths.items():
        os.environ[variable] = str(path)

    # Create cache directories only. Do not create or copy a token file.
    for path in (
        HPC_CACHE_ROOT,
        HF_HOME_DIR,
        HF_HOME_DIR / "hub",
        HF_HOME_DIR / "transformers",
        HF_HOME_DIR / "datasets",
        HPC_CACHE_ROOT / "torch",
    ):
        path.mkdir(parents=True, exist_ok=True)

    # In Jupyter, huggingface_hub may have been imported in an earlier cell. Refresh
    # its cached module-level paths so restarting the kernel is usually unnecessary.
    hf_constants = sys.modules.get("huggingface_hub.constants")
    if hf_constants is not None:
        for name in (
            "HF_HOME",
            "HF_HUB_CACHE",
            "HUGGINGFACE_HUB_CACHE",
            "HF_TOKEN_PATH",
        ):
            if name in cache_paths:
                setattr(hf_constants, name, str(cache_paths[name]))


configure_hpc_caches()

# ------------------------------
# Logging
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
CHOSEN_FOLD: int = 1
CHOSEN_CKPT: str = "dpn68_new_fold1_epoch52_0.0001_1.0000.ckpt"

CAM_METHOD: str = "eigencam"          # "gradcam", "eigencam", ...
ABNORMAL_CLASS_INDEX: int = 1         # ignored by class-agnostic EigenCAM
SAVE_CONTOURS: bool = True

# Threshold used only for masks, contours, and bounding boxes.
# The heatmap overlay always uses the continuous CAM.
THRESHOLD_CAM: float = 0.7

# Seamless-overlay controls. COLORMAP_TURBO produces the smooth
# blue/cyan/yellow/red progression shown in the reference image.
COLORMAP: int = getattr(cv2, "COLORMAP_TURBO", cv2.COLORMAP_JET)
OVERLAY_MAX_ALPHA: float = 0.42
OVERLAY_FADE_START: float = 0.05
OVERLAY_FULL_STRENGTH: float = 0.85
OVERLAY_OPACITY_GAMMA: float = 1.10
CAM_BLUR_SIGMA_FRACTION: float = 0.012
PRESERVE_XRAY_LUMINANCE: bool = True

# Suppress tiny contour fragments without changing the continuous overlay.
MIN_CONTOUR_AREA_FRACTION: float = 0.0005

CONTOUR_COLOR: Tuple[int, int, int] = (0, 0, 255)  # BGR red
CONTOUR_THICKNESS: int = 3
CONTOUR_LINE_TYPE: int = cv2.LINE_AA

IMAGE_EXTS: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")

# ------------------------------
# Preprocessing
# ------------------------------
def build_preprocess(img_size: int) -> transforms.Compose:
    """Resize -> normalize -> tensor (deterministic for visualization)."""
    return transforms.Compose(
        [
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )

def preprocess_image(orig_rgb: np.ndarray, preprocess: transforms.Compose, device: torch.device) -> torch.Tensor:
    """orig_rgb in [0,1] float32, H x W x 3."""
    img_u8 = (np.clip(orig_rgb, 0.0, 1.0) * 255).astype(np.uint8)
    return preprocess(img_u8).unsqueeze(0).to(device)

# ------------------------------
# Target-layer selection
# ------------------------------
def _kernel_tuple(m: nn.Conv2d) -> Tuple[int, int]:
    if isinstance(m.kernel_size, tuple):
        return (m.kernel_size[0], m.kernel_size[1])
    return (int(m.kernel_size), int(m.kernel_size))

def _last_conv_kgt1(module: nn.Module) -> Optional[nn.Conv2d]:
    """Last Conv2d with kernel > 1; if none, last Conv2d."""
    last_any: Optional[nn.Conv2d] = None
    last_kgt1: Optional[nn.Conv2d] = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            last_any = m
            k = _kernel_tuple(m)
            if max(k) > 1:
                last_kgt1 = m
    return last_kgt1 if last_kgt1 is not None else last_any

def _vit_patch_embed_conv(module: nn.Module) -> Optional[nn.Conv2d]:
    """For timm ViTs: try patch_embed conv."""
    candidates: List[Tuple[str, nn.Conv2d]] = []
    for name, m in module.named_modules():
        if isinstance(m, nn.Conv2d) and ("patch_embed" in name or "patch" in name):
            candidates.append((name, m))
    return candidates[-1][1] if candidates else None

def select_target_layer_for_cam(model: nn.Module) -> nn.Conv2d:
    """
    Priority:
      1) model.post3x3 with kernel 3 x 3
      2) last Conv2d (k>1) in backbone
      3) ViT patch-embed Conv2d
      4) last Conv2d anywhere
    """
    post3x3 = getattr(model, "post3x3", None)
    if isinstance(post3x3, nn.Sequential):
        for m in post3x3.modules():
            if isinstance(m, nn.Conv2d) and _kernel_tuple(m) == (3, 3):
                return m
    elif isinstance(post3x3, nn.Conv2d):
        if _kernel_tuple(post3x3) == (3, 3):
            return post3x3

    if hasattr(model, "backbone"):
        target = _last_conv_kgt1(model.backbone)  # type: ignore[attr-defined]
        if isinstance(target, nn.Conv2d):
            return target
        target = _vit_patch_embed_conv(model.backbone)  # type: ignore[attr-defined]
        if isinstance(target, nn.Conv2d):
            return target

    target = _last_conv_kgt1(model)
    if isinstance(target, nn.Conv2d):
        return target

    raise RuntimeError("No suitable Conv2d layer found for CAM.")

def _find_module_name(root: nn.Module, target: nn.Module) -> Optional[str]:
    for name, m in root.named_modules():
        if m is target:
            return name or "<root>"
    return None

# ------------------------------
# Model loading (strict -> fallback)
# ------------------------------
def load_model_auto_strict(ckpt_path: str, h: Dict[str, object], device: torch.device) -> nn.Module:
    """Load the notebook's TBModel checkpoint and return model.eval() on device."""
    model_class = globals().get("TBModel")
    if model_class is None:
        raise NameError(
            "TBModel is not defined. Import/define the same TBModel class used for "
            "training, then run this script from Jupyter with `%run -i`."
        )

    try:
        model = model_class.load_from_checkpoint(
            ckpt_path, h=h, strict=True, map_location=device
        )
        logging.info("[LOAD] strict=True worked.")
    except Exception as exc:
        logging.warning(
            "[LOAD] strict=True failed: %s | retrying with strict=False.", exc
        )
        model = model_class.load_from_checkpoint(
            ckpt_path, h=h, strict=False, map_location=device
        )
        logging.info("[LOAD] strict=False used.")
    return model.to(device).eval()

# ------------------------------
# Split handling# ------------------------------
@dataclass(frozen=True)
class SplitSpec:
    name: str
    root: Path

def collect_normal_not_images(split_root: Path) -> List[Path]:
    """
    Collect images for ONLY the normal_not class.

    Expected:
        split_root/normal_not/*.ext
    Fallback:
        any file under split_root where a path component == "normal_not"
    """
    preferred = split_root / "normal_not"
    images: List[Path] = []

    if preferred.exists():
        for p in preferred.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                images.append(p)
    else:
        for p in split_root.rglob("*"):
            if not (p.is_file() and p.suffix.lower() in IMAGE_EXTS):
                continue
            if "normal_not" in {part.lower() for part in p.parts}:
                images.append(p)

    images = sorted(images)
    if not images:
        raise RuntimeError(
            f"No normal_not images found under split root: {split_root} "
            f"(checked {preferred} and recursive 'normal_not' path filter)."
        )
    return images

# ------------------------------
# CAM post-processing and seamless overlay
# ------------------------------
def normalize_cam_to_unit(cam_map: np.ndarray, upper_percentile: float = 99.5) -> np.ndarray:
    """Robustly normalize one CAM to [0, 1] without propagating NaN/Inf."""
    cam = np.nan_to_num(cam_map, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
    cam = cam - float(np.min(cam))
    if not np.any(cam > 0):
        return np.zeros_like(cam, dtype=np.float32)

    scale = float(np.percentile(cam, upper_percentile))
    if scale <= np.finfo(np.float32).eps:
        return np.zeros_like(cam, dtype=np.float32)
    return np.clip(cam / scale, 0.0, 1.0).astype(np.float32)

def resize_and_smooth_cam(
    cam01: np.ndarray,
    output_size: Tuple[int, int],
    sigma_fraction: float,
) -> np.ndarray:
    """Resize CAM to (width, height), softly blur it, and return [0, 1]."""
    width, height = output_size
    resized = cv2.resize(cam01, (width, height), interpolation=cv2.INTER_CUBIC)
    resized = np.clip(resized, 0.0, 1.0).astype(np.float32)

    sigma = max(height, width) * max(0.0, float(sigma_fraction))
    if sigma > 0:
        resized = cv2.GaussianBlur(
            resized,
            ksize=(0, 0),
            sigmaX=sigma,
            sigmaY=sigma,
            borderType=cv2.BORDER_REPLICATE,
        )
    return normalize_cam_to_unit(resized, upper_percentile=100.0)

def build_soft_alpha_map(
    cam01: np.ndarray,
    max_alpha: float,
    fade_start: float,
    full_strength: float,
    opacity_gamma: float,
) -> np.ndarray:
    """Convert CAM intensity into a smooth per-pixel opacity map."""
    if not 0.0 <= fade_start < full_strength <= 1.0:
        raise ValueError("Require 0 <= fade_start < full_strength <= 1.")
    if not 0.0 <= max_alpha <= 1.0:
        raise ValueError("max_alpha must be in [0, 1].")
    if opacity_gamma <= 0:
        raise ValueError("opacity_gamma must be > 0.")

    x = np.clip((cam01 - fade_start) / (full_strength - fade_start), 0.0, 1.0)
    x = x * x * (3.0 - 2.0 * x)  # smoothstep: no abrupt opacity edge
    return (float(max_alpha) * np.power(x, float(opacity_gamma))).astype(np.float32)

def make_seamless_overlay(
    bgr: np.ndarray,
    cam01: np.ndarray,
    colormap: int,
    max_alpha: float,
    fade_start: float,
    full_strength: float,
    opacity_gamma: float,
    preserve_luminance: bool,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Blend a continuous color CAM with spatially varying opacity.

    Pixels below fade_start remain exactly equal to the source radiograph.
    When preserve_luminance=True, the overlay changes chroma while restoring
    the source LAB lightness within the activated region, retaining anatomy.
    """
    cam_u8 = np.uint8(np.rint(np.clip(cam01, 0.0, 1.0) * 255.0))
    heat_bgr = cv2.applyColorMap(cam_u8, int(colormap))
    alpha = build_soft_alpha_map(
        cam01,
        max_alpha=max_alpha,
        fade_start=fade_start,
        full_strength=full_strength,
        opacity_gamma=opacity_gamma,
    )

    a3 = alpha[..., None]
    blended = np.rint(
        bgr.astype(np.float32) * (1.0 - a3)
        + heat_bgr.astype(np.float32) * a3
    ).clip(0, 255).astype(np.uint8)

    if preserve_luminance and np.any(alpha > 0):
        base_lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        blend_lab = cv2.cvtColor(blended, cv2.COLOR_BGR2LAB)
        blend_lab[..., 0] = base_lab[..., 0]
        luminance_preserved = cv2.cvtColor(blend_lab, cv2.COLOR_LAB2BGR)

        # Avoid LAB round-trip changes where no overlay is present.
        active = alpha > 0
        blended[active] = luminance_preserved[active]

    return blended, heat_bgr, alpha

def make_threshold_outputs(cam01: np.ndarray, threshold: float) -> Tuple[np.ndarray, np.ndarray]:
    """Return an intensity-preserving thresholded mask and a binary mask."""
    threshold = float(threshold)
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be in [0, 1].")
    binary = np.uint8(cam01 >= threshold) * 255
    masked = np.where(binary > 0, cam01, 0.0)
    masked_u8 = np.uint8(np.rint(np.clip(masked, 0.0, 1.0) * 255.0))
    return masked_u8, binary

# ------------------------------
# Core routine
# ------------------------------
def run_gradcam_train_val_test(
    h: Dict[str, object],
    chosen_fold: int,
    chosen_ckpt_name: str,
    method: str = "eigencam",
    abnormal_class_index: int = 1,
    save_contours: bool = True,
    threshold_cam: float = 0.7,
    colormap: int = COLORMAP,
    overlay_max_alpha: float = 0.42,
    overlay_fade_start: float = 0.05,
    overlay_full_strength: float = 0.85,
    overlay_opacity_gamma: float = 1.10,
    cam_blur_sigma_fraction: float = 0.012,
    preserve_xray_luminance: bool = True,
    min_contour_area_fraction: float = 0.0005,
) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = str(h["model"])
    img_size = int(h["img_size"])
    preprocess = build_preprocess(img_size)

    ckpt_dir = Path(str(h["output_dir"])) / f"fold_{chosen_fold}" / model_id
    ckpt_path = ckpt_dir / chosen_ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Chosen checkpoint not found: {ckpt_path}")

    model = load_model_auto_strict(str(ckpt_path), h=h, device=device)

    target_layer = select_target_layer_for_cam(model)
    target_name = _find_module_name(model, target_layer) or "<unnamed>"
    logging.info(
        "[CAM] Target layer: %s | kernel=%s", target_name, _kernel_tuple(target_layer)
    )

    cam_classes = {
        "gradcam": GradCAM,
        "gradcam++": GradCAMPlusPlus,
        "scorecam": ScoreCAM,
        "ablationcam": AblationCAM,
        "xgradcam": XGradCAM,
        "layercam": LayerCAM,
        "fullgrad": FullGrad,
        "eigencam": EigenCAM,
        "eigengradcam": EigenGradCAM,
        "hirescam": HiResCAM,
    }
    method_name = method.lower().strip()
    if method_name not in cam_classes:
        raise ValueError(
            f"Unknown CAM method '{method}'. Options: {sorted(cam_classes.keys())}"
        )
    if method_name == "eigencam":
        logging.info(
            "[CAM] EigenCAM is class-agnostic; abnormal_class_index is not used."
        )

    cam = cam_classes[method_name](
        model=model,
        target_layers=[target_layer],
        reshape_transform=None,
    )
    cam.batch_size = 1

    # Preserve the existing output-path convention. The threshold in the folder
    # name refers only to masks/contours; overlays use the continuous CAM.
    out_root = (
        Path(str(h["output_dir"]))
        / model_id
        / f"{method_name}_train_val_test_fold{chosen_fold}__thr{threshold_cam:.2f}"
    )
    out_root.mkdir(parents=True, exist_ok=True)

    splits: List[SplitSpec] = [
        SplitSpec("train", Path(str(h["train_dir"]))),
        SplitSpec("val", Path(str(h["val_dir"]))),
        SplitSpec("test", Path(str(h["test_dir"]))),
    ]

    logging.info(
        "[OVERLAY] continuous CAM | colormap=%s | max_alpha=%.2f | "
        "fade_start=%.2f | full_strength=%.2f | preserve_luminance=%s",
        colormap,
        overlay_max_alpha,
        overlay_fade_start,
        overlay_full_strength,
        preserve_xray_luminance,
    )
    logging.info(
        "[MASK] threshold_cam=%.3f (used only for masks/contours/bboxes)",
        threshold_cam,
    )
    logging.info(
        "[CAM] Each map is normalized independently; colors represent relative "
        "importance within an image, not magnitude across images."
    )

    try:
        for split in splits:
            image_paths = collect_normal_not_images(split.root)
            logging.info(
                "[SPLIT] %s: %d normal_not images found under %s",
                split.name,
                len(image_paths),
                split.root,
            )

            base = out_root / split.name
            (base / "masks").mkdir(parents=True, exist_ok=True)
            (base / "heatmaps").mkdir(parents=True, exist_ok=True)
            if save_contours:
                (base / "contours").mkdir(parents=True, exist_ok=True)
                (base / "bboxes").mkdir(parents=True, exist_ok=True)

            for index, image_path in enumerate(image_paths, start=1):
                # IMREAD_COLOR keeps the same 8-bit BGR input behavior as the
                # original script and therefore does not alter model inference.
                bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                if bgr is None:
                    logging.warning("[WARN] Could not read image: %s", image_path)
                    continue

                height, width = bgr.shape[:2]
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
                input_tensor = preprocess_image(rgb, preprocess, device)

                if method_name == "eigencam":
                    cam_raw = cam(input_tensor=input_tensor)[0]
                else:
                    cam_raw = cam(
                        input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(int(abnormal_class_index))],
                        aug_smooth=True,
                        eigen_smooth=True,
                    )[0]

                # Normalize at model resolution, then interpolate and smooth at
                # native image resolution. Thresholding occurs only afterward.
                cam_small = normalize_cam_to_unit(cam_raw)
                cam_full = resize_and_smooth_cam(
                    cam_small,
                    output_size=(width, height),
                    sigma_fraction=cam_blur_sigma_fraction,
                )

                thresholded_mask, binary = make_threshold_outputs(
                    cam_full, threshold=threshold_cam
                )

                tag = (
                    f"{split.name}__{image_path.stem}__{model_id}__"
                    f"fold{chosen_fold}__{method_name}"
                )

                cv2.imwrite(
                    str(base / "masks" / f"{tag}__mask.png"), thresholded_mask
                )

                overlay, _, _ = make_seamless_overlay(
                    bgr=bgr,
                    cam01=cam_full,
                    colormap=colormap,
                    max_alpha=overlay_max_alpha,
                    fade_start=overlay_fade_start,
                    full_strength=overlay_full_strength,
                    opacity_gamma=overlay_opacity_gamma,
                    preserve_luminance=preserve_xray_luminance,
                )
                cv2.imwrite(str(base / "heatmaps" / f"{tag}__heat.png"), overlay)

                if save_contours:
                    kernel = np.ones((3, 3), dtype=np.uint8)
                    clean_binary = cv2.morphologyEx(
                        binary, cv2.MORPH_OPEN, kernel, iterations=1
                    )
                    clean_binary = cv2.morphologyEx(
                        clean_binary, cv2.MORPH_CLOSE, kernel, iterations=1
                    )

                    contours, _ = cv2.findContours(
                        clean_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                    )
                    min_area = max(0.0, float(min_contour_area_fraction)) * height * width
                    contours = [
                        contour
                        for contour in contours
                        if cv2.contourArea(contour) >= min_area
                    ]

                    contour_image = bgr.copy()
                    cv2.drawContours(
                        contour_image,
                        contours,
                        -1,
                        CONTOUR_COLOR,
                        CONTOUR_THICKNESS,
                        lineType=CONTOUR_LINE_TYPE,
                    )
                    cv2.imwrite(
                        str(base / "contours" / f"{tag}__contours.png"),
                        contour_image,
                    )

                    bbox_image = bgr.copy()
                    for contour in contours:
                        x, y, box_width, box_height = cv2.boundingRect(contour)
                        cv2.rectangle(
                            bbox_image,
                            (x, y),
                            (x + box_width, y + box_height),
                            CONTOUR_COLOR,
                            CONTOUR_THICKNESS,
                            lineType=CONTOUR_LINE_TYPE,
                        )
                    cv2.imwrite(
                        str(base / "bboxes" / f"{tag}__bboxes.png"), bbox_image
                    )

                if index % 100 == 0 or index == len(image_paths):
                    logging.info(
                        "[%s] Processed %d/%d",
                        split.name,
                        index,
                        len(image_paths),
                    )
    finally:
        if hasattr(cam, "activations_and_grads"):
            try:
                cam.activations_and_grads.release()
            except Exception:
                pass
        del cam, model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    logging.info("Done. Outputs saved under: %s", out_root)

# ------------------------------
# Execute
# ------------------------------
def main() -> None:
    run_gradcam_train_val_test(
        h=h,
        chosen_fold=CHOSEN_FOLD,
        chosen_ckpt_name=CHOSEN_CKPT,
        method=CAM_METHOD,
        abnormal_class_index=ABNORMAL_CLASS_INDEX,
        save_contours=SAVE_CONTOURS,
        threshold_cam=THRESHOLD_CAM,
        colormap=COLORMAP,
        overlay_max_alpha=OVERLAY_MAX_ALPHA,
        overlay_fade_start=OVERLAY_FADE_START,
        overlay_full_strength=OVERLAY_FULL_STRENGTH,
        overlay_opacity_gamma=OVERLAY_OPACITY_GAMMA,
        cam_blur_sigma_fraction=CAM_BLUR_SIGMA_FRACTION,
        preserve_xray_luminance=PRESERVE_XRAY_LUMINANCE,
        min_contour_area_fraction=MIN_CONTOUR_AREA_FRACTION,
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Overlay radiologist bounding boxes on existing seamless Grad-CAM images.

"""

from __future__ import annotations

import csv
import logging
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import cv2

# =============================================================================
# Configuration
# =============================================================================

GRADCAM_ROOT = Path(
    "/path/to/eigencam/overlaid/images"
)
ANNOTATION_ROOT = Path(
    "/path/to/radiologist_specific_annotations"
)

# Keep derived images beside the CAM products without altering existing outputs.
OUTPUT_ROOT = GRADCAM_ROOT / "radiologist_box_overlays"

SPLITS: Tuple[str, ...] = ("train", "val", "test")
READERS: Tuple[str, ...] = ("rad1", "rad2", "rad3")
MODEL_ID = "dpn68_new"
FOLD = 1
CAM_METHOD = "eigencam"

# Bright colors with a dark halo remain visible on grayscale and Turbo heatmaps.
# Values are OpenCV BGR tuples.
READER_COLORS: Mapping[str, Tuple[int, int, int]] = {
    "rad1": (0, 255, 255),     # yellow
    "rad2": (255, 255, 0),    # cyan
    "rad3": (255, 0, 255),   # magenta
}

# Dashed boxes reduce obstruction. Reader-specific phases help preserve different
# colors when boxes overlap in the combined image.
DRAW_DASHED_BOXES = True
BOX_THICKNESS = 3
HALO_THICKNESS = 2
DASH_LENGTH = 14
GAP_LENGTH = 8
READER_DASH_PHASE: Mapping[str, int] = {
    "rad1": 0,
    "rad2": 7,
    "rad3": 14,
}

# Labels can obscure anatomy, so they are disabled by default. Descriptions remain
# available in the source text files and the output manifest.
DRAW_LESION_LABELS = False
FONT_SCALE = 0.55
FONT_THICKNESS = 1

# Do not make a missing annotation look like an explicit negative assessment.
SAVE_IMAGES_WITHOUT_BOXES = False
OVERWRITE_EXISTING = True
LOG_EVERY_N = 100

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("radiologist_box_overlay")

Box = Tuple[int, int, int, int]
@dataclass(frozen=True)
class Annotation:
    box: Box
    label: str = ""

@dataclass
class ReaderSummary:
    annotation_files: int = 0
    images_with_boxes: int = 0
    boxes_drawn: int = 0
    missing_files: int = 0
    empty_files: int = 0
    malformed_lines: int = 0
    duplicate_boxes: int = 0
    clipped_boxes: int = 0
    rejected_boxes: int = 0
    outputs_written: int = 0

def validate_configuration() -> None:
    """Fail early and identify any missing study resource."""
    if not GRADCAM_ROOT.is_dir():
        raise FileNotFoundError(f"Grad-CAM root not found: {GRADCAM_ROOT}")
    if not ANNOTATION_ROOT.is_dir():
        raise FileNotFoundError(f"Annotation root not found: {ANNOTATION_ROOT}")

    missing_heatmap_dirs = [
        GRADCAM_ROOT / split / "heatmaps"
        for split in SPLITS
        if not (GRADCAM_ROOT / split / "heatmaps").is_dir()
    ]
    if missing_heatmap_dirs:
        joined = "\n  ".join(str(path) for path in missing_heatmap_dirs)
        raise FileNotFoundError(f"Missing heatmap directories:\n  {joined}")

    missing_reader_dirs = [
        ANNOTATION_ROOT / reader / "txt"
        for reader in READERS
        if not (ANNOTATION_ROOT / reader / "txt").is_dir()
    ]
    if missing_reader_dirs:
        joined = "\n  ".join(str(path) for path in missing_reader_dirs)
        raise FileNotFoundError(f"Missing radiologist annotation directories:\n  {joined}")

    missing_styles = set(READERS) - set(READER_COLORS)
    if missing_styles:
        raise ValueError(f"No display color configured for: {sorted(missing_styles)}")

def build_annotation_index(reader: str) -> Dict[str, Path]:
    """Index TXT files case-insensitively and reject ambiguous identifiers."""
    txt_root = ANNOTATION_ROOT / reader / "txt"
    index: Dict[str, Path] = {}
    for path in sorted(txt_root.rglob("*.txt")):
        key = path.stem.casefold()
        if key in index and index[key] != path:
            raise RuntimeError(
                f"Duplicate annotation identifier for {reader}: {index[key]} and {path}"
            )
        index[key] = path
    return index

def iter_heatmaps(split: str) -> List[Path]:
    """Return only final seamless heatmap overlays for a split."""
    folder = GRADCAM_ROOT / split / "heatmaps"
    return sorted(path for path in folder.glob("*__heat.png") if path.is_file())

def image_id_from_heatmap(path: Path, split: str) -> str:
    """Extract the original CXR stem even when it contains underscores."""
    prefix = f"{split}__"
    suffix = f"__{MODEL_ID}__fold{FOLD}__{CAM_METHOD}__heat"
    stem = path.stem
    if not stem.startswith(prefix) or not stem.endswith(suffix):
        raise ValueError(
            f"Unexpected heatmap name: {path.name}; "
            f"expected {prefix}<image-id>{suffix}.png"
        )
    image_id = stem[len(prefix) : len(stem) - len(suffix)]
    if not image_id:
        raise ValueError(f"Empty image identifier parsed from: {path.name}")
    return image_id

def _coordinate_tokens(line: str) -> Tuple[List[str], str]:
    """Support 'x1 y1 x2 y2 label' and 'reader x1 y1 x2 y2 label'."""
    tokens = line.split()
    if tokens and tokens[0].casefold() in {name.casefold() for name in READERS}:
        tokens = tokens[1:]
    if len(tokens) < 4:
        return tokens, ""
    return tokens[:4], " ".join(tokens[4:]).strip()

def read_annotations(path: Path) -> Tuple[List[Annotation], int, int]:
    """Read, validate, and deduplicate boxes; return annotations and issue counts."""
    annotations_by_box: Dict[Box, Annotation] = {}
    malformed = 0
    duplicates = 0

    try:
        lines = path.read_text(encoding="utf-8-sig").splitlines()
    except UnicodeDecodeError:
        lines = path.read_text(encoding="latin-1").splitlines()

    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        coordinate_tokens, label = _coordinate_tokens(line)
        if len(coordinate_tokens) != 4:
            malformed += 1
            log.warning("Malformed annotation skipped: %s:%d", path, line_number)
            continue
        try:
            values = [float(token) for token in coordinate_tokens]
        except ValueError:
            malformed += 1
            log.warning("Non-numeric coordinates skipped: %s:%d", path, line_number)
            continue
        if not all(math.isfinite(value) for value in values):
            malformed += 1
            log.warning("Non-finite coordinates skipped: %s:%d", path, line_number)
            continue

        x1, y1, x2, y2 = (int(round(value)) for value in values)
        x1, x2 = sorted((x1, x2))
        y1, y2 = sorted((y1, y2))
        box = (x1, y1, x2, y2)

        if box in annotations_by_box:
            duplicates += 1
            prior = annotations_by_box[box]
            prior_labels = {part.strip().casefold() for part in prior.label.split("/")}
            if label and label.casefold() not in prior_labels:
                merged = " / ".join(part for part in (prior.label, label) if part)
                annotations_by_box[box] = Annotation(box=box, label=merged)
            continue
        annotations_by_box[box] = Annotation(box=box, label=label)

    return list(annotations_by_box.values()), malformed, duplicates

def clamp_annotation(
    annotation: Annotation, width: int, height: int
) -> Tuple[Optional[Annotation], bool]:
    """Clamp a partly visible box and reject boxes outside or without positive area."""
    x1, y1, x2, y2 = annotation.box
    if x2 < 0 or y2 < 0 or x1 >= width or y1 >= height:
        return None, False

    clamped: Box = (
        max(0, min(x1, width - 1)),
        max(0, min(y1, height - 1)),
        max(0, min(x2, width - 1)),
        max(0, min(y2, height - 1)),
    )
    was_clipped = clamped != annotation.box
    cx1, cy1, cx2, cy2 = clamped
    if cx2 <= cx1 or cy2 <= cy1:
        return None, was_clipped
    return Annotation(box=clamped, label=annotation.label), was_clipped

def _segment_points(
    start: Tuple[int, int],
    end: Tuple[int, int],
    dash: int,
    gap: int,
    phase: int,
) -> Iterable[Tuple[Tuple[int, int], Tuple[int, int]]]:
    """Yield dashed-line segments along an arbitrary edge."""
    x0, y0 = start
    x1, y1 = end
    dx, dy = x1 - x0, y1 - y0
    length = math.hypot(dx, dy)
    if length == 0:
        return

    cycle = dash + gap
    position = -float(phase % cycle)
    while position < length:
        segment_start = max(position, 0.0)
        segment_end = min(position + dash, length)
        if segment_end > segment_start:
            p0 = (
                int(round(x0 + dx * segment_start / length)),
                int(round(y0 + dy * segment_start / length)),
            )
            p1 = (
                int(round(x0 + dx * segment_end / length)),
                int(round(y0 + dy * segment_end / length)),
            )
            yield p0, p1
        position += cycle

def _draw_line_with_halo(
    image,
    start: Tuple[int, int],
    end: Tuple[int, int],
    color: Tuple[int, int, int],
) -> None:
    outer_thickness = BOX_THICKNESS + 2 * HALO_THICKNESS
    if HALO_THICKNESS > 0:
        cv2.line(image, start, end, (20, 20, 20), outer_thickness, cv2.LINE_AA)
    cv2.line(image, start, end, color, BOX_THICKNESS, cv2.LINE_AA)

def draw_box(image, box: Box, color: Tuple[int, int, int], phase: int) -> None:
    x1, y1, x2, y2 = box
    corners = ((x1, y1), (x2, y1), (x2, y2), (x1, y2))
    edges = tuple(zip(corners, corners[1:] + corners[:1]))

    if DRAW_DASHED_BOXES:
        for edge_index, (start, end) in enumerate(edges):
            edge_phase = phase + edge_index * (DASH_LENGTH + GAP_LENGTH) // 4
            for segment_start, segment_end in _segment_points(
                start, end, DASH_LENGTH, GAP_LENGTH, edge_phase
            ):
                _draw_line_with_halo(image, segment_start, segment_end, color)
    else:
        outer_thickness = BOX_THICKNESS + 2 * HALO_THICKNESS
        if HALO_THICKNESS > 0:
            cv2.rectangle(
                image, (x1, y1), (x2, y2), (20, 20, 20),
                outer_thickness, cv2.LINE_AA,
            )
        cv2.rectangle(image, (x1, y1), (x2, y2), color, BOX_THICKNESS, cv2.LINE_AA)

def draw_label(image, annotation: Annotation, color: Tuple[int, int, int]) -> None:
    if not DRAW_LESION_LABELS or not annotation.label:
        return
    x1, y1, _, _ = annotation.box
    text = annotation.label
    (text_width, text_height), baseline = cv2.getTextSize(
        text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, FONT_THICKNESS
    )
    image_height, image_width = image.shape[:2]
    text_x = max(0, min(x1, image_width - text_width - 4))
    text_y = y1 - 7 if y1 - text_height - baseline - 8 >= 0 else y1 + text_height + 8
    top = max(0, text_y - text_height - 4)
    bottom = min(image_height - 1, text_y + baseline + 3)
    cv2.rectangle(
        image, (text_x, top), (text_x + text_width + 4, bottom), (20, 20, 20), -1
    )
    cv2.putText(
        image, text, (text_x + 2, text_y), cv2.FONT_HERSHEY_SIMPLEX,
        FONT_SCALE, color, FONT_THICKNESS, cv2.LINE_AA,
    )

def draw_annotations(image, annotations: Sequence[Annotation], reader: str):
    """Return a copy; the source heatmap array is never modified in place."""
    output = image.copy()
    color = READER_COLORS[reader]
    phase = READER_DASH_PHASE.get(reader, 0)
    for annotation in annotations:
        draw_box(output, annotation.box, color=color, phase=phase)
        draw_label(output, annotation, color=color)
    return output

def write_png(path: Path, image) -> bool:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not OVERWRITE_EXISTING:
        return False
    if not cv2.imwrite(str(path), image, [cv2.IMWRITE_PNG_COMPRESSION, 3]):
        raise OSError(f"OpenCV failed to write: {path}")
    return True

def run_pipeline() -> None:
    validate_configuration()
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    annotation_indexes = {reader: build_annotation_index(reader) for reader in READERS}
    summaries = {reader: ReaderSummary() for reader in READERS}
    for reader in READERS:
        summaries[reader].annotation_files = len(annotation_indexes[reader])

    manifest_rows: List[Dict[str, object]] = []
    total_heatmaps = combined_outputs = unreadable_heatmaps = unparsable_heatmaps = 0
    missing_examples: Dict[str, List[str]] = {reader: [] for reader in READERS}

    log.info("Grad-CAM root: %s", GRADCAM_ROOT)
    log.info("Annotation root: %s", ANNOTATION_ROOT)
    log.info("Output root: %s", OUTPUT_ROOT)
    log.info("Using existing seamless __heat.png images; no CAM reprocessing will occur.")

    for split in SPLITS:
        heatmaps = iter_heatmaps(split)
        total_heatmaps += len(heatmaps)
        log.info("[%s] Found %d heatmaps", split, len(heatmaps))

        for position, heatmap_path in enumerate(heatmaps, start=1):
            try:
                image_id = image_id_from_heatmap(heatmap_path, split)
            except ValueError as exc:
                unparsable_heatmaps += 1
                log.warning("Skipping heatmap: %s", exc)
                continue

            base_image = cv2.imread(str(heatmap_path), cv2.IMREAD_COLOR)
            if base_image is None:
                unreadable_heatmaps += 1
                log.warning("Unreadable heatmap skipped: %s", heatmap_path)
                continue
            height, width = base_image.shape[:2]

            valid_by_reader: Dict[str, List[Annotation]] = {}
            txt_by_reader: Dict[str, Optional[Path]] = {}
            for reader in READERS:
                summary = summaries[reader]
                txt_path = annotation_indexes[reader].get(image_id.casefold())
                txt_by_reader[reader] = txt_path
                if txt_path is None:
                    summary.missing_files += 1
                    if len(missing_examples[reader]) < 20:
                        missing_examples[reader].append(f"{split}:{image_id}")
                    valid_by_reader[reader] = []
                    continue

                annotations, malformed, duplicates = read_annotations(txt_path)
                summary.malformed_lines += malformed
                summary.duplicate_boxes += duplicates
                valid_annotations: List[Annotation] = []
                for annotation in annotations:
                    valid, was_clipped = clamp_annotation(annotation, width, height)
                    if valid is None:
                        summary.rejected_boxes += 1
                        continue
                    if was_clipped:
                        summary.clipped_boxes += 1
                    valid_annotations.append(valid)

                if valid_annotations:
                    summary.images_with_boxes += 1
                    summary.boxes_drawn += len(valid_annotations)
                else:
                    summary.empty_files += 1
                valid_by_reader[reader] = valid_annotations

            # Per-reader views preserve each radiologist's exact box geometry.
            for reader in READERS:
                annotations = valid_by_reader[reader]
                if not annotations and not SAVE_IMAGES_WITHOUT_BOXES:
                    continue
                rendered = draw_annotations(base_image, annotations, reader)
                output_name = f"{heatmap_path.stem}__{reader}__boxes.png"
                output_path = OUTPUT_ROOT / split / reader / output_name
                if write_png(output_path, rendered):
                    summaries[reader].outputs_written += 1
                manifest_rows.append({
                    "split": split,
                    "image_id": image_id,
                    "reader": reader,
                    "boxes": len(annotations),
                    "annotation_file": str(txt_by_reader[reader] or ""),
                    "source_heatmap": str(heatmap_path),
                    "output_image": str(output_path),
                })

            # Combined views are supplementary; use per-reader outputs when boxes overlap.
            combined = base_image.copy()
            total_combined_boxes = 0
            for reader in READERS:
                annotations = valid_by_reader[reader]
                if annotations:
                    combined = draw_annotations(combined, annotations, reader)
                    total_combined_boxes += len(annotations)

            if total_combined_boxes or SAVE_IMAGES_WITHOUT_BOXES:
                combined_name = f"{heatmap_path.stem}__all_readers__boxes.png"
                combined_path = OUTPUT_ROOT / split / "combined" / combined_name
                if write_png(combined_path, combined):
                    combined_outputs += 1
                manifest_rows.append({
                    "split": split,
                    "image_id": image_id,
                    "reader": "combined",
                    "boxes": total_combined_boxes,
                    "annotation_file": "",
                    "source_heatmap": str(heatmap_path),
                    "output_image": str(combined_path),
                })

            if position % LOG_EVERY_N == 0 or position == len(heatmaps):
                log.info("[%s] Processed %d/%d", split, position, len(heatmaps))

    manifest_path = OUTPUT_ROOT / "overlay_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        fieldnames = (
            "split", "image_id", "reader", "boxes", "annotation_file",
            "source_heatmap", "output_image",
        )
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(manifest_rows)

    log.info(
        "Processed %d heatmaps; unreadable=%d; unparsable names=%d",
        total_heatmaps, unreadable_heatmaps, unparsable_heatmaps,
    )
    for reader in READERS:
        summary = summaries[reader]
        log.info(
            "[%s] TXT=%d | images with boxes=%d | boxes=%d | outputs=%d | "
            "missing TXT=%d | empty/invalid TXT=%d | duplicates removed=%d | "
            "clipped=%d | rejected=%d | malformed lines=%d",
            reader, summary.annotation_files, summary.images_with_boxes,
            summary.boxes_drawn, summary.outputs_written, summary.missing_files,
            summary.empty_files, summary.duplicate_boxes, summary.clipped_boxes,
            summary.rejected_boxes, summary.malformed_lines,
        )
        if missing_examples[reader]:
            log.warning(
                "[%s] First missing annotations: %s",
                reader, ", ".join(missing_examples[reader]),
            )

    log.info("Combined overlays written: %d", combined_outputs)
    log.info("Manifest: %s", manifest_path)
    log.info("Completed. Outputs: %s", OUTPUT_ROOT)

def main() -> None:
    run_pipeline()

if __name__ == "__main__":
    main()

## END OF CODE